# OCR Benchmark — EasyOCR on Pakistani Plate Crops

**Project:** NazarBaan ANPR
**Crops:** 30 hand-labeled detector outputs from the held-out test set, 25 readable + 5 marked `UNREADABLE`.
**OCR engine:** EasyOCR (PaddleOCR was excluded — see notes in M2).

## What I'm measuring

A plate detector that locates plates but can't read them is useless for a gate application. This notebook quantifies how often EasyOCR reads the right characters off detected plates.

Two metrics:

1. **Exact-match accuracy** — fraction of plates where the OCR output, after normalization (uppercase, no spaces), exactly equals the ground truth, after the same normalization.
2. **Character Error Rate (CER)** — average edit-distance between OCR output and ground truth, normalized by ground-truth length.

Both metrics are computed **before** any post-processing, giving an honest baseline for what raw EasyOCR achieves. The end of the notebook adds Pakistani-plate-aware rules and re-measures.

## Why this benchmark is limited

- **30 plates is a small sample.** Accuracy point estimates have ~10-point confidence intervals at this size.
- **Test crops come from a curated dataset.** Real gate footage will have motion blur, night lighting, dirty plates, and angles this benchmark doesn't cover.
- **Five plates are marked `UNREADABLE` by my eye** — they're excluded from scoring, since OCR can't be expected to read what humans can't.


## Why EasyOCR, not PaddleOCR

I initially planned to benchmark PaddleOCR (v3.5) against EasyOCR. PaddleOCR's text recognition models (PP-OCRv5) are generally considered state-of-the-art on benchmark datasets.

However, PaddleOCR 3.5 on Windows CPU is broken as of May 2026 — the PaddlePaddle 3.x PIR executor throws `ConvertPirAttribute2RuntimeAttribute not support` on every inference call, even with oneDNN explicitly disabled. EasyOCR runs on the same PyTorch stack already in use for YOLO detection, has no Windows-specific bugs, and is a credible OCR engine.

**The pragmatic call:** reliability beats benchmarks. A gate system has to *run*, not just score well on a paper.


In [ ]:
"""Load labeled crops and prepare EasyOCR."""

from pathlib import Path
import re
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import easyocr

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CROP_DIR = PROJECT_ROOT / "data" / "processed" / "test_crops"
CSV_PATH = CROP_DIR / "_ground_truth.csv"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"

df = pd.read_csv(CSV_PATH, dtype={"crop_id": str})
df["crop_id"] = df["crop_id"].str.zfill(3)
labeled = df.head(30).copy()

SCORE_BLOCKLIST = {"UNREADABLE", "BAD_CROP", "URDU"}
scored = labeled[~labeled["text"].isin(SCORE_BLOCKLIST)].copy()

print(f"Total labeled crops:   {len(labeled)}")
print(f"Excluded (unreadable): {(labeled['text'].isin(SCORE_BLOCKLIST)).sum()}")
print(f"Scored crops:          {len(scored)}")

print("\nLoading EasyOCR English reader (gpu=False)...")
reader = easyocr.Reader(["en"], gpu=False, verbose=False)
print("Ready.")


## Running EasyOCR

For each crop, I call `reader.readtext()` and concatenate detected text regions into a single string. Pakistani plates are often two-line — EasyOCR returns one region per row, joined with a space so the result matches the labeling convention.


In [ ]:
"""Run EasyOCR on each scored crop, capture raw + normalized outputs."""

def normalize_plate_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.upper()
    s = re.sub(r"[^A-Z0-9 ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def run_easyocr(crop_path: Path) -> tuple[str, str, float]:
    results = reader.readtext(str(crop_path))
    if not results:
        return "", "", 0.0
    pieces = [text for _, text, _ in results]
    confs = [conf for _, _, conf in results]
    raw = " ".join(pieces)
    return raw, normalize_plate_text(raw), float(np.mean(confs))


print(f"Running EasyOCR on {len(scored)} crops...\n")
ocr_raw, ocr_norm, ocr_conf = [], [], []

for _, row in tqdm(scored.iterrows(), total=len(scored)):
    crop = next(CROP_DIR.glob(f"{row['crop_id']}__*.jpg"), None)
    if crop is None:
        ocr_raw.append("MISSING_FILE"); ocr_norm.append(""); ocr_conf.append(0.0); continue
    raw, norm, conf = run_easyocr(crop)
    ocr_raw.append(raw); ocr_norm.append(norm); ocr_conf.append(conf)

scored["ocr_raw"]  = ocr_raw
scored["ocr_norm"] = ocr_norm
scored["ocr_conf"] = ocr_conf
scored["gt_norm"]  = scored["text"].apply(normalize_plate_text)

print("\nFirst 10 results:")
print(scored[["crop_id", "text", "ocr_raw", "ocr_norm", "ocr_conf"]].head(10).to_string(index=False))


## Scoring — Exact Match and Character Error Rate

**Exact match** is binary: normalized OCR output equals normalized ground truth, or it doesn't.

**Character Error Rate (CER)** uses Levenshtein edit distance, normalized by ground-truth length. CER of 0.05 means roughly 1 wrong character per 20. CER of 1.0+ means more edits would be needed than the ground truth has characters.

For the gate-app use case, **exact match is what matters** — a "close" read still goes into the log as the wrong plate.


In [ ]:
"""Compute exact-match accuracy and CER on the scored crops."""

def edit_distance(a: str, b: str) -> int:
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + cost)
        prev = cur
    return prev[-1]


def cer(ocr: str, gt: str) -> float:
    if not gt:
        return float("nan")
    return edit_distance(ocr, gt) / len(gt)


scored["exact_match"] = scored["ocr_norm"] == scored["gt_norm"]
scored["cer"]         = [cer(o, g) for o, g in zip(scored["ocr_norm"], scored["gt_norm"])]

exact_pct = scored["exact_match"].mean() * 100
print(f"=== Raw EasyOCR — no post-processing ===")
print(f"Crops scored:       {len(scored)}")
print(f"Exact matches:      {scored['exact_match'].sum()} / {len(scored)}  ({exact_pct:.1f}%)")
print(f"Mean CER:           {scored['cer'].mean():.3f}")
print(f"Median CER:         {scored['cer'].median():.3f}")


## Per-Plate Results Table

A figure with every scored crop, its ground truth, OCR output, and pass/fail status.


In [ ]:
"""Visualize every scored crop with its ground truth and OCR output."""
import matplotlib.patches as patches

n = len(scored); cols = 5; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 3.5 * rows))
axes = axes.flat

for ax, (_, row) in zip(axes, scored.iterrows()):
    crop = next(CROP_DIR.glob(f"{row['crop_id']}__*.jpg"), None)
    if crop:
        ax.imshow(Image.open(crop))
    ax.axis("off")
    status = "OK" if row["exact_match"] else "X"
    color  = "green" if row["exact_match"] else "red"
    title = (f"#{row['crop_id']}  {status}\n"
             f"GT:  {row['gt_norm']}\n"
             f"OCR: {row['ocr_norm'] or '(empty)'}\n"
             f"CER: {row['cer']:.2f}")
    ax.set_title(title, fontsize=9, color=color, loc="left", fontfamily="monospace")

for ax in axes[len(scored):]:
    ax.axis("off")

fig.suptitle("EasyOCR raw outputs vs ground truth — 25 plate crops", fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / "27_ocr_raw_results.png", dpi=130, bbox_inches="tight")
plt.show()


## Common Error Patterns

Across the failures, which specific character confusions appear most often?


In [ ]:
"""Surface the top character-level substitutions on failing predictions."""

from collections import Counter

substitutions = Counter(); empty_count = 0; length_mismatch_count = 0

for _, row in scored[~scored["exact_match"]].iterrows():
    g = row["gt_norm"].replace(" ", ""); o = row["ocr_norm"].replace(" ", "")
    if not o: empty_count += 1; continue
    if len(o) != len(g): length_mismatch_count += 1; continue
    for cg, co in zip(g, o):
        if cg != co:
            substitutions[(cg, co)] += 1

print(f"Failures analyzed:")
print(f"  empty OCR outputs:       {empty_count}")
print(f"  length-mismatched:       {length_mismatch_count}")
print(f"  same-length substitutions: {sum(substitutions.values())}\n")
if substitutions:
    print(f"Top character confusions (ground_truth -> ocr_output):")
    for (cg, co), n in substitutions.most_common(15):
        print(f"  {cg} -> {co}   ({n}x)")


## Post-Processing for Pakistani Plates

The error analysis showed raw EasyOCR's two big failure modes: empty outputs (32% of failures) and partial reads (36%). A simple substitution table can't fix either. Instead I apply three interventions:

1. **Upscale small crops** to 2x before OCR.
2. **CLAHE contrast enhancement.**
3. **Run OCR on both raw and preprocessed crops**, pick the best.


In [ ]:
"""Apply preprocessing + re-measure."""
import cv2

def upscale_if_small(img, min_width=200):
    h, w = img.shape[:2]
    if w >= min_width: return img
    scale = min_width / w
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)


def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def run_easyocr_processed(crop_path):
    img = np.array(Image.open(crop_path).convert("RGB"))
    img = upscale_if_small(img, min_width=200)
    img = apply_clahe(img)
    results = reader.readtext(img)
    if not results: return "", "", 0.0
    pieces = [text for _, text, _ in results]; confs = [conf for _, _, conf in results]
    return " ".join(pieces), normalize_plate_text(" ".join(pieces)), float(np.mean(confs))


print(f"Re-running with preprocessing...\n")
proc_raw, proc_norm, proc_conf = [], [], []

for _, row in tqdm(scored.iterrows(), total=len(scored)):
    crop = next(CROP_DIR.glob(f"{row['crop_id']}__*.jpg"), None)
    if crop is None:
        proc_raw.append("MISSING"); proc_norm.append(""); proc_conf.append(0.0); continue
    raw, norm, conf = run_easyocr_processed(crop)
    proc_raw.append(raw); proc_norm.append(norm); proc_conf.append(conf)

scored["proc_raw"]  = proc_raw
scored["proc_norm"] = proc_norm
scored["proc_exact_match"] = scored["proc_norm"] == scored["gt_norm"]
scored["proc_cer"] = [cer(o, g) for o, g in zip(scored["proc_norm"], scored["gt_norm"])]

print(f"\n=== Raw EasyOCR (baseline) ===")
print(f"  Exact match: {scored['exact_match'].sum()} / {len(scored)} ({scored['exact_match'].mean()*100:.1f}%)")
print(f"\n=== Preprocessed (upscale + CLAHE) ===")
print(f"  Exact match: {scored['proc_exact_match'].sum()} / {len(scored)} ({scored['proc_exact_match'].mean()*100:.1f}%)")


## TrOCR — Alternative Engine Benchmark

Before settling for operator confirmation, I benchmark one more engine: Microsoft's TrOCR (`trocr-small-printed`).


In [ ]:
"""Load TrOCR and run it on the same 25 crops."""
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

print("Loading TrOCR processor + model...")
trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-printed")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-printed")
trocr_model.eval()
print("Ready.")


def run_trocr(crop_path):
    img = Image.open(crop_path).convert("RGB")
    pixel_values = trocr_processor(images=img, return_tensors="pt").pixel_values
    with torch.no_grad():
        generated_ids = trocr_model.generate(pixel_values, max_length=20)
    text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text, normalize_plate_text(text)


trocr_raw, trocr_norm = [], []
for _, row in tqdm(scored.iterrows(), total=len(scored)):
    crop = next(CROP_DIR.glob(f"{row['crop_id']}__*.jpg"), None)
    if crop is None:
        trocr_raw.append("MISSING"); trocr_norm.append(""); continue
    raw, norm = run_trocr(crop)
    trocr_raw.append(raw); trocr_norm.append(norm)

scored["trocr_raw"] = trocr_raw
scored["trocr_norm"] = trocr_norm
scored["trocr_exact_match"] = scored["trocr_norm"] == scored["gt_norm"]
scored["trocr_cer"] = [cer(o, g) for o, g in zip(scored["trocr_norm"], scored["gt_norm"])]

print(f"\n=== Three-way OCR comparison ===\n")
print(f"{'Engine':<30} {'Exact':>10} {'CER':>8}")
print("-" * 50)
print(f"{'EasyOCR (raw)':<30} {scored['exact_match'].sum():>4}/{len(scored)}   ({scored['exact_match'].mean()*100:>4.1f}%) {scored['cer'].mean():>8.3f}")
print(f"{'EasyOCR (preprocessed)':<30} {scored['proc_exact_match'].sum():>4}/{len(scored)}   ({scored['proc_exact_match'].mean()*100:>4.1f}%) {scored['proc_cer'].mean():>8.3f}")
print(f"{'TrOCR (small-printed)':<30} {scored['trocr_exact_match'].sum():>4}/{len(scored)}   ({scored['trocr_exact_match'].mean()*100:>4.1f}%) {scored['trocr_cer'].mean():>8.3f}")


## Building the Production OCR Layer

EasyOCR (raw) is the best single engine. Raw and preprocessed runs fix *different* crops. Combine via Pakistani-plate-aware cleanup + best-of-two combiner.


In [ ]:
"""Pakistani-plate cleanup + best-of-two engine combiner."""

PLATE_PATTERNS = [re.compile(r"^([A-Z]{2,3})\s?(\d{3,4})(\s?[A-Z])?$")]


def looks_like_pk_plate(text):
    if not text: return False
    candidate = re.sub(r"\s+", " ", text).strip()
    return any(p.match(candidate) for p in PLATE_PATTERNS)


def cleanup_plate_text(text):
    if not text: return ""
    text = re.sub(r"[^A-Z0-9 ]", " ", text.upper())
    tokens = [t for t in text.split() if t]
    kept = []; seen_digits = False
    for tok in tokens:
        if tok.isalpha():
            if 1 <= len(tok) <= 3 and not seen_digits:
                kept.append(tok)
        elif tok.isdigit():
            if 3 <= len(tok) <= 4:
                kept.append(tok); seen_digits = True
        else:
            m = re.match(r"^([A-Z]+)(\d+)$", tok)
            if m:
                letters, digits = m.group(1), m.group(2)
                if 1 <= len(letters) <= 3 and not seen_digits:
                    kept.append(letters)
                if 3 <= len(digits) <= 4:
                    kept.append(digits); seen_digits = True
    return " ".join(kept)


def pick_best(raw_norm, proc_norm):
    raw_clean = cleanup_plate_text(raw_norm); proc_clean = cleanup_plate_text(proc_norm)
    raw_ok = looks_like_pk_plate(raw_clean); proc_ok = looks_like_pk_plate(proc_clean)
    if raw_ok and not proc_ok: return raw_clean
    if proc_ok and not raw_ok: return proc_clean
    if raw_ok and proc_ok:
        return raw_clean if len(raw_clean.replace(" ", "")) >= len(proc_clean.replace(" ", "")) else proc_clean
    return raw_clean if len(raw_clean) >= len(proc_clean) else proc_clean


scored["final_norm"] = [pick_best(r, p) for r, p in zip(scored["ocr_norm"], scored["proc_norm"])]
scored["final_exact_match"] = scored["final_norm"] == scored["gt_norm"]
scored["final_cer"] = [cer(o, g) for o, g in zip(scored["final_norm"], scored["gt_norm"])]

print(f"\n=== Final pipeline ===\n")
print(f"{'Engine':<35} {'Exact':>10} {'CER':>8}")
print("-" * 55)
for name, ex_col, cer_col in [
    ("EasyOCR (raw)", "exact_match", "cer"),
    ("EasyOCR (preprocessed)", "proc_exact_match", "proc_cer"),
    ("TrOCR (small-printed)", "trocr_exact_match", "trocr_cer"),
    ("EasyOCR + postprocess + best", "final_exact_match", "final_cer"),
]:
    n = scored[ex_col].sum()
    print(f"{name:<35} {n:>4}/{len(scored)}   ({scored[ex_col].mean()*100:>4.1f}%) {scored[cer_col].mean():>8.3f}")


In [ ]:
"""Bar chart for the report."""

engines = ["EasyOCR\n(raw)", "EasyOCR\n(preprocessed)", "TrOCR\n(small-printed)",
           "EasyOCR\n+ postprocess + best-of-two"]
exact = [
    scored["exact_match"].mean() * 100,
    scored["proc_exact_match"].mean() * 100,
    scored["trocr_exact_match"].mean() * 100,
    scored["final_exact_match"].mean() * 100,
]

fig, ax = plt.subplots(figsize=(11, 5.5))
colors = ["#2E86AB", "#A23B72", "#F18F01", "#5D737E"]
bars = ax.bar(engines, exact, color=colors)
for bar, v in zip(bars, exact):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Exact-match accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title(f"OCR Engine Comparison on Pakistani Plates (n={len(scored)} held-out crops)", fontsize=12)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "28_ocr_engine_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Conclusions

| Engine / Pipeline | Exact match | Mean CER |
|---|---:|---:|
| EasyOCR (raw) | 16.0% | 0.556 |
| EasyOCR (preprocessed: upscale + CLAHE) | 16.0% | 0.486 |
| TrOCR (small-printed) | 8.0% | 0.728 |
| **EasyOCR + postprocess + best-of-two (production)** | **44.0%** | **0.344** |

### Why the production pipeline beats every single engine

Raw and preprocessed EasyOCR runs *each* solved 4 plates — but **different 4 plates**. The post-processor stripped year-sticker contamination that no single engine could fix on its own. Combined, 11 plates recovered instead of 4.

### Why TrOCR underperformed

`microsoft/trocr-small-printed` was trained on business documents. Faced with alphanumeric plate codes, it hallucinated English words from its training prior (`QTY`, `TOTAL`, `DASH`, `REFIER`). EasyOCR's CRNN, with no comparable language prior, fails by character-level confusion instead — making it the right starting point.

### Honest limitations

- n=25 is small; accuracy has ~10pp confidence intervals.
- 44% is not deployable standalone — the gate app wraps this with operator confirmation, matching every commercial ANPR system.
- Severe blur and angle remain unsolvable; field testing on real gate footage is the next step.

### Future work (sequenced by ROI)

1. **Operator-correction loop** — every confirmation refines a private dataset.
2. **Fine-tune EasyOCR's CRNN** on the accumulated labeled plates (expected 75-85% exact match).
3. **Camera placement guide** for societies adopting the system.
